# SeamlessM4T v2 Large: Myanmar ASR with MLflow

Fine-tune `facebook/seamless-m4t-v2-large` using the combined audio-backed catalog. This follows the corresponding experiment under `scripts/training/`, with a shared reader for original ASR/UCSY files and downloaded FLEURS/YODAS audio. Unlabeled VOA is excluded.

Read [the training guide](../README.md) first. Start Docker MLflow from a terminal, select a separate GPU Python kernel, and run top to bottom. `QUICK_RUN=True` is a short training check that still loads this full model. It is not a full evaluation or a small-model substitute. Run only one of the three notebooks at a time per GPU.

## 1. Locate the repository and install notebook dependencies

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents)
             if (p / "scripts/data/build_combined_catalog.py").is_file()), None)
assert ROOT is not None, "Open Jupyter from inside the myanmar-asr repository"
print("Repository:", ROOT)

Repository: D:\Development\myanmar-asr


In [2]:
# Install PyTorch >=2.6 for your GPU first; see ../README.md.
%pip install -q -r {ROOT / "notebook/requirements-training.txt"}
# If package versions changed, restart the kernel and rerun from the first cell.

Note: you may need to restart the kernel to use updated packages.


## 2. Configure this experiment

Use a new `RUN_LABEL` for changed model, data, or hyperparameters. Quick and full runs use separate directories. The defaults freeze the acoustic encoder and use batch size one plus gradient accumulation. Large-model memory requirements still depend on GPU, precision, and audio/label lengths.

In [3]:
import importlib
import traceback

packages = [
    "numpy",
    "torch",
    "transformers",
    "mlflow",
]

for package in packages:
    print(f"\nTesting: {package}")
    try:
        module = importlib.import_module(package)
        print(f"PASS: {package} {getattr(module, '__version__', '')}")
    except Exception:
        print(f"FAILED: {package}")
        traceback.print_exc()
        break


Testing: numpy
PASS: numpy 2.5.4

Testing: torch
PASS: torch 2.11.0+cu128

Testing: transformers


d:\Development\myanmar-asr\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PASS: transformers 4.57.1

Testing: mlflow
PASS: mlflow 2.19.0


In [ ]:
import os
import json
import shutil
import hashlib
import platform
import numpy as np
import torch
import transformers
import mlflow
from transformers import Seq2SeqTrainingArguments, EarlyStoppingCallback, set_seed

sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebook"))
from training_common import (
    load_rows, file_digest, filter_labels, CatalogSpeechDataset, SpeechCollator,
    SpeechTrainer, metric_function, MLflowMetricsCallback, latest_checkpoint,
    tracked_run, score_texts, decode_predictions, save_predictions,
)

MODEL_ID = 'facebook/seamless-m4t-v2-large'
MODEL_REVISION = "main"  # Resolved to a commit before loading model and processor.
FAMILY = 'seamless'
QUICK_RUN = False
RUN_LABEL = "baseline"
FREEZE_ENCODER = True
SEED = 42
LEARNING_RATE = 5e-05
EPOCHS = 3
TRAIN_BATCH_SIZE = 1
GRAD_ACCUM = 4 if QUICK_RUN else 16
EVAL_SAVE_STEPS = 5 if QUICK_RUN else 100
MAX_STEPS = 10 if QUICK_RUN else -1
MAX_GENERATION_LENGTH = 448
MIN_FREE_GB = 60
LOG_MODEL_ARTIFACTS = False  # Local best_model is saved; uploading weights duplicates disk use.
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5050")
EXPERIMENT_NAME = "myanmar-asr-notebooks"
CATALOG_DIR = ROOT / "combined/myanmar_asr_all"
RUN_DIR = ROOT / "notebook/runs/seamless" / ("quick" if QUICK_RUN else "full") / RUN_LABEL

set_seed(SEED)
print({"model": MODEL_ID, "quick": QUICK_RUN, "run_dir": str(RUN_DIR),
       "torch": torch.__version__, "transformers": transformers.__version__,
       "cuda": torch.cuda.is_available(), "mlflow": MLFLOW_URI})

{'model': 'facebook/seamless-m4t-v2-large', 'quick': False, 'run_dir': 'D:\\Development\\myanmar-asr\\notebook\\runs\\seamless\\full\\baseline', 'torch': '2.11.0+cu128', 'transformers': '4.57.1', 'cuda': True, 'mlflow': 'http://127.0.0.1:5050'}


## 3. Check catalog, tracking, and storage before downloading weights

The catalog's `audio_data/` and `raw/` source links must resolve. These notebooks read encoded Hugging Face audio through SoundFile and resample on demand, so TorchCodec is not needed. Only labeled, curated clips of at most 30 seconds are accepted. Existing source/video and inferred-speaker overlap remains; see the catalog report before claiming independent speaker/video evaluation.

MLflow connection errors stop the notebook. This avoids silently training without tracking. The Docker service proxies artifacts, including configuration, metrics and prediction CSVs.

In [5]:
assert transformers.__version__ == "4.57.1", "Restart into the installed training kernel"
from packaging.version import Version
assert Version(torch.__version__.split("+")[0]) >= Version("2.6"), "Install PyTorch >=2.6 for checkpoint loading"
assert (CATALOG_DIR / "summary.json").is_file(), "Build the combined catalog first"
assert (CATALOG_DIR / "audio_data").is_dir() and (CATALOG_DIR / "raw").is_dir(), "Restore catalog source links"
mlflow.set_tracking_uri(MLFLOW_URI)
client = mlflow.MlflowClient()
client.search_experiments(max_results=1)
print("MLflow is reachable:", MLFLOW_URI)

train_file = CATALOG_DIR / "train_whisper.csv"
validation_file = CATALOG_DIR / "validation_whisper.csv"
train_rows = load_rows(train_file, limit=64 if QUICK_RUN else None, seed=SEED)
validation_rows = load_rows(validation_file, limit=16 if QUICK_RUN else None, seed=SEED)
test_files = sorted(CATALOG_DIR.glob("test_*_whisper.csv"))
assert test_files, "No held-out test manifests found"
catalog_summary = json.loads((CATALOG_DIR / "summary.json").read_text(encoding="utf-8"))
print("Selected train / validation:", len(train_rows), len(validation_rows))
print("Held-out tests (full mode only):", [p.stem for p in test_files])
print("Known record/video overlap:", catalog_summary["curated_overlap"]["record_group"])

free_gb = shutil.disk_usage(ROOT).free / 1024**3
print(f"Free disk: {free_gb:.1f} GiB; minimum preflight: {MIN_FREE_GB} GiB")
if free_gb < MIN_FREE_GB:
    raise RuntimeError("Free more disk space before downloading model weights/checkpoints")
if not QUICK_RUN and not torch.cuda.is_available():
    raise RuntimeError("Full runs target a CUDA GPU. Select the GPU kernel/machine first.")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

MLflow is reachable: http://127.0.0.1:5050
Selected train / validation: 21805 1894
Held-out tests (full mode only): ['test_asr_whisper', 'test_fleurs_whisper', 'test_ucsy_whisper', 'test_user_study_whisper', 'test_yodas_my_whisper']
Known record/video overlap: {'test_yodas_my / train': 20, 'test_yodas_my / validation': 17, 'train / validation': 27}
Free disk: 1792.3 GiB; minimum preflight: 60 GiB
GPU: NVIDIA GeForce RTX 5060 Ti VRAM GiB: 15.9


### MLflow UI inside Jupyter

This frame uses your browser to reach the tracking URI. If it cannot load, open the printed URL in a separate tab.

In [6]:
from IPython.display import IFrame, display
print('MLflow UI:', MLFLOW_URI)
display(IFrame(src=MLFLOW_URI, width='100%', height=520))

MLflow UI: http://127.0.0.1:5050


## 4. Load the model and Myanmar processor

In [7]:
from huggingface_hub import HfApi
resolved_revision = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION).sha

from transformers import AutoProcessor, SeamlessM4Tv2ForSpeechToText
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=resolved_revision)
processor.tokenizer.src_lang = "mya"
processor.tokenizer.tgt_lang = "mya"
model = SeamlessM4Tv2ForSpeechToText.from_pretrained(
    MODEL_ID, revision=resolved_revision, dtype=torch.float32)
if FREEZE_ENCODER:
    for parameter in model.speech_encoder.parameters():
        parameter.requires_grad = False
GENERATION_KWARGS = {"tgt_lang": "mya"}
MAX_GENERATION_LENGTH = min(MAX_GENERATION_LENGTH, model.config.max_position_embeddings)

model.config.use_cache = False
model.generation_config.use_cache = True
model.generation_config.max_new_tokens = None
model.generation_config.max_length = MAX_GENERATION_LENGTH
model.generation_config.num_beams = 1
total_parameters = sum(p.numel() for p in model.parameters())
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
# Two resumable checkpoints, final weights, optimizer moments, and a small reserve.
estimated_gb = (3 * total_parameters * 4 + 2 * trainable_parameters * 8) / 1024**3 + 5
free_gb = shutil.disk_usage(ROOT).free / 1024**3
print({"resolved_revision": resolved_revision, "parameters": total_parameters,
       "trainable_parameters": trainable_parameters, "checkpoint_space_estimate_gib": round(estimated_gb, 1)})
if free_gb < estimated_gb:
    raise RuntimeError("Insufficient estimated space for checkpoints/final model; move the run to a larger disk")

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 11.18it/s]


{'resolved_revision': '5f8cc790b19fc3f67a61c105133b20b34e3dcb76', 'parameters': 1501842240, 'trainable_parameters': 866795520, 'checkpoint_space_estimate_gib': 34.7}


## 5. Prepare lazy audio datasets and check label lengths

Labels exceeding the model/generation limit are reported and excluded from training/validation, never truncated. Counts can differ by tokenizer. Held-out test references remain complete: test inference does not feed labels into the model, so all test clips in the duration views can be scored.

In [8]:
decoder_start = model.config.decoder_start_token_id
train_rows, rejected_train = filter_labels(train_rows, processor, FAMILY, decoder_start, MAX_GENERATION_LENGTH)
validation_rows, rejected_validation = filter_labels(validation_rows, processor, FAMILY, decoder_start, MAX_GENERATION_LENGTH)
print({"train": len(train_rows), "validation": len(validation_rows),
       "rejected_train_labels": len(rejected_train), "rejected_validation_labels": len(rejected_validation)})
train_dataset = CatalogSpeechDataset(train_rows, CATALOG_DIR, processor.feature_extractor)
validation_dataset = CatalogSpeechDataset(validation_rows, CATALOG_DIR, processor.feature_extractor)
collator = SpeechCollator(FAMILY, processor.tokenizer.pad_token_id, decoder_start)
sample_batch = collator([train_dataset[0]])
print({key: tuple(value.shape) for key, value in sample_batch.items()})
assert sample_batch["labels"].shape[0] == 1

{'train': 21805, 'validation': 1894, 'rejected_train_labels': 0, 'rejected_validation_labels': 0}
{'input_features': (1, 204, 160), 'attention_mask': (1, 204), 'labels': (1, 23), 'decoder_input_ids': (1, 23)}


## 6. Configure Trainer and immutable run metadata

CER with whitespace removed selects the best checkpoint. Raw CER and whitespace-delimited WER are also logged as percentages. Myanmar WER depends strongly on spacing and is not a linguistically segmented word score. The shared trainer supplies the correct language during scheduled evaluation as well as explicit prediction.

In [9]:
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16 and torch.cuda.get_device_capability(0)[0] >= 7
training_args = Seq2SeqTrainingArguments(
    output_dir=str(RUN_DIR),
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.06,
    weight_decay=0.05,
    label_smoothing_factor=0.1,
    num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=use_bf16,
    fp16=use_fp16,
    optim="adamw_torch",
    eval_strategy="steps",
    eval_steps=EVAL_SAVE_STEPS,
    save_strategy="steps",
    save_steps=EVAL_SAVE_STEPS,
    save_total_limit=2,
    save_safetensors=True,
    logging_steps=1 if QUICK_RUN else 10,
    predict_with_generate=True,
    generation_max_length=MAX_GENERATION_LENGTH,
    generation_num_beams=1,
    eval_accumulation_steps=4,
    load_best_model_at_end=True,
    metric_for_best_model="cer_no_space",
    greater_is_better=False,
    dataloader_num_workers=0,
    dataloader_pin_memory=torch.cuda.is_available(),
    remove_unused_columns=False,
    report_to=[],  # One explicit MLflow callback; the notebook owns the run lifetime.
    push_to_hub=False,
    seed=SEED,
)
manifest_hashes = {p.name: file_digest(p) for p in [train_file, validation_file, *test_files]}
selection_hash = hashlib.sha256(json.dumps({
    "train": [r["sample_id"] for r in train_rows],
    "validation": [r["sample_id"] for r in validation_rows],
}, sort_keys=True).encode()).hexdigest()
configuration = {
    "model_id": MODEL_ID, "model_revision": resolved_revision, "family": FAMILY,
    "quick_run": QUICK_RUN, "run_label": RUN_LABEL, "freeze_encoder": FREEZE_ENCODER,
    "train_clips": len(train_rows), "validation_clips": len(validation_rows),
    "rejected_train_labels": len(rejected_train), "rejected_validation_labels": len(rejected_validation),
    "train_hours": sum(float(r["duration_s"]) for r in train_rows) / 3600,
    "validation_hours": sum(float(r["duration_s"]) for r in validation_rows) / 3600,
    "selection_sha256": selection_hash, "catalog_summary_sha256": file_digest(CATALOG_DIR / "summary.json"),
    "manifest_sha256": manifest_hashes,
    "seed": SEED, "learning_rate": LEARNING_RATE, "epochs": EPOCHS,
    "batch_size": TRAIN_BATCH_SIZE, "gradient_accumulation": GRAD_ACCUM,
    "max_steps": MAX_STEPS, "eval_save_steps": EVAL_SAVE_STEPS,
    "generation_max_length": MAX_GENERATION_LENGTH,
    "precision": "bf16" if use_bf16 else "fp16" if use_fp16 else "fp32",
    "warmup_ratio": 0.06, "weight_decay": 0.05, "label_smoothing": 0.1,
    "metric_for_best_model": "cer_no_space", "scoring": "NFC; collapsed whitespace; CER additionally without spaces",
}
trainer = SpeechTrainer(
    model=model, args=training_args,
    train_dataset=train_dataset, eval_dataset=validation_dataset,
    data_collator=collator, compute_metrics=metric_function(processor),
    processing_class=processor.feature_extractor,
    generation_kwargs=GENERATION_KWARGS,
    callbacks=[MLflowMetricsCallback(), EarlyStoppingCallback(early_stopping_patience=3)],
)
print("Effective training batch:", TRAIN_BATCH_SIZE * GRAD_ACCUM)

Effective training batch: 16


## 7. Resume or train with live MLflow tracking

Open the displayed MLflow run link to watch loss, learning rate, validation WER/CER, and CUDA peak memory. Run settings, label rejections and catalog statistics are artifacts. Weights stay local by default.

On interruption, restart the kernel and rerun with identical settings and `RUN_LABEL`. The newest checkpoint whose state archives and model headers pass checks is selected; a corrupted optimizer ZIP is skipped. If none passes, the notebook stops. The saved MLflow run ID is reused. Work after the latest usable checkpoint must be repeated. These checks cannot repair corruption or guarantee every state will deserialize.

In [ ]:
checkpoint = latest_checkpoint(RUN_DIR)
if checkpoint and not (RUN_DIR / "run_config.json").is_file():
    raise RuntimeError("Checkpoint has no matching notebook run configuration; choose a new RUN_LABEL")
print("Resuming from:" if checkpoint else "Starting in:", checkpoint or RUN_DIR)
with tracked_run(RUN_DIR, MLFLOW_URI, EXPERIMENT_NAME,
                 f"{MODEL_ID.split('/')[-1]}-{RUN_LABEL}-{'quick' if QUICK_RUN else 'full'}",
                 configuration) as active:
    RUN_ID = active.info.run_id
    print("MLflow run:", f"{MLFLOW_URI}/#/experiments/{active.info.experiment_id}/runs/{RUN_ID}")
    mlflow.log_dict(catalog_summary, "dataset/catalog_summary.json")
    mlflow.log_dict({"train": rejected_train, "validation": rejected_validation},
                    "dataset/rejected_labels.json")
    mlflow.log_dict(training_args.to_dict(), "configuration/trainer_arguments.json")
    train_result = trainer.train(resume_from_checkpoint=str(checkpoint) if checkpoint else None)
    trainer.save_model(str(RUN_DIR / "best_model"))
    processor.save_pretrained(str(RUN_DIR / "best_model"))
    validation_metrics = trainer.evaluate(metric_key_prefix="validation")
    mlflow.log_dict(train_result.metrics, "metrics/training.json")
    mlflow.log_dict(validation_metrics, "metrics/validation.json")
    mlflow.set_tag("best_checkpoint", str(trainer.state.best_model_checkpoint))
    mlflow.set_tag("local_best_model", str(RUN_DIR / "best_model"))
    if LOG_MODEL_ARTIFACTS:
        mlflow.log_artifacts(str(RUN_DIR / "best_model"), artifact_path="best_model")
print("Training:", train_result.metrics)
print("Validation:", validation_metrics)

Starting in: D:\Development\myanmar-asr\notebook\runs\seamless\full\baseline
MLflow run: http://127.0.0.1:5050/#/experiments/1/runs/9b424b89246244d78ccad23c5ac1ec7f


Step,Training Loss,Validation Loss


## 8. Evaluate each held-out test separately (full runs only)

The selected best model generates predictions for all clips in each test duration view. Complete references are scored without teacher-forced test labels or token truncation. Prediction CSVs and per-test JSON metrics are saved locally and uploaded into the same MLflow run. Quick checks skip held-out tests.

In [ ]:
if QUICK_RUN:
    print("Quick run: held-out tests skipped. Use a new full run for final evaluation.")
else:
    test_results = {}
    with mlflow.start_run(run_id=RUN_ID):
        for path in test_files:
            split_name = path.stem.removesuffix("_whisper")
            rows = load_rows(path)
            test_dataset = CatalogSpeechDataset(rows, CATALOG_DIR, processor.feature_extractor,
                                                include_labels=False)
            result = trainer.predict(test_dataset, metric_key_prefix=split_name)
            hypotheses = decode_predictions(processor, result.predictions)
            metrics = {f"{split_name}_{key}": value for key, value in
                       score_texts([r["text"] for r in rows], hypotheses).items()}
            metrics[f"{split_name}_clips"] = len(rows)
            mlflow.log_metrics(metrics, step=trainer.state.global_step)
            predictions_path = RUN_DIR / f"{split_name}_predictions.csv"
            save_predictions(predictions_path, rows, hypotheses)
            mlflow.log_artifact(str(predictions_path), artifact_path="predictions")
            test_results[split_name] = metrics
            print(split_name, metrics)
        result_path = RUN_DIR / "test_metrics.json"
        result_path.write_text(json.dumps(test_results, indent=2), encoding="utf-8")
        mlflow.log_artifact(str(result_path), artifact_path="metrics")

Quick run: held-out tests skipped. Use a new full run for final evaluation.
